##Домашнее задание №3. Классификация сайтов 18+ по текстовому описанию средствами Classic ML.

Все сайты 18+, которые приведены в качестве примеров разбора - зацензурены, чтобы данный ноутбук можно было разместить в репозитории.

Данные представляют собой обезличенные заголовки веб-страниц из открытых источников. Любые совпадения с реальными названиями случайны и являются частью публичного веб-архива.

In [ ]:
import numpy as np
import pandas as pd
!pip install pymorphy2
import json
import re
import requests
import spacy
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from bs4 import BeautifulSoup
import pymorphy2
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing  import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score
from IPython.core.display import HTML, display

SEED = 42
TOKEN_PATTERN = '[а-яёa-z]+'

In [ ]:
train_df = pd.read_csv("train.csv")
train_df

,ID,url,title,label
0,0,m.kp.md,"Экс-министр экономики Молдовы - главе МИДЭИ, ц...",0
1,1,www.kp.by,Эта песня стала известна многим телезрителям б...,0
2,2,fanserials.tv,Банши 4 сезон 2 серия Бремя красоты смотреть о...,0
3,3,colorbox.spb.ru,Не Беси Меня Картинки,0
4,4,tula-sport.ru,В Новомосковске сыграют следж-хоккеисты алекси...,0
...,...,...,...,...
135304,135304,mail.ru,пора тюльпанов турецкий сериал на русском язык...,0
135305,135305,www.ntv.ru,Остросюжетный сериал «Шеф. Игра на повышение»....,0
135306,135306,topclassiccarsforsale.com,"1941 Plymouth Special Deluxe Hot Rod, Automati...",0
135307,135307,wowcream.ru,Купить It's Skin Сыворотка питательная Power 1...,0


In [ ]:
test_df = pd.read_csv("test.csv")

**Введение.**

В данной работе строится модель для обнаружения сайтов (18+).
Сразу стоит обратить внимание, что такой контент, как правило, выделяется специфическими словами. Причём эти слова как правило друг с другом связаны плохо.

Например, описание сайта может представлять собой безграмотно написанный текст, или вообще рандомное сочетание слов-маркеров в произвольном порядке. Понятно, что методы построения модели, учитывающие контекст, будут работать плохо.

Поэтому в данной работе за основу построения моделей обучения будет браться метод CountVectorizer, который максимально не учитывает контекст. Но для этого метода очевидно нужна хорошая подготовка текста, чтобы максимально хорошо выделить те самые специфические слова, выдающие контент (18+)

**Начало работы**

1) Для начала сделаем минимальную обработку текста. Для обучения берём пока только тексты titl'ов. Уберём стоп-слова как русские, так и английские. Токенизируем тексты titl'ов в корпусах по словам. Обработка самих слов отсутствует.

In [ ]:
#обучающие выборки (x_train_t, y_train_t) и валидационные выборки (X_test_t, Y_test_t)
title_train_t = train_df["title"].values.astype('U')
label_train_t = train_df["label"].values
x_train_t, X_test_t, y_train_t, Y_test_t = train_test_split(title_train_t, label_train_t, test_size=0.5, random_state=SEED)

In [ ]:
#берём русские и английские стоп-слова из библиотеки nltk
stop_words_ru_1 = nltk.corpus.stopwords.words('russian')
stop_words_en_1 = nltk.corpus.stopwords.words('english')
stop_words_1 = stop_words_ru_1 + stop_words_en_1

count_model_1 = Pipeline([
    (
        'vectorizer',
        CountVectorizer(
            stop_words=stop_words_1, ngram_range=(1, 2), token_pattern=TOKEN_PATTERN,
            min_df=3, max_df=0.8, lowercase=True
        )
    ),
    ('clf', SGDClassifier(random_state=SEED, loss='log_loss', class_weight='balanced'))
])
count_model_1.fit(x_train_t, y_train_t)
print("Train score:", f1_score(y_train_t, count_model_1.predict(x_train_t)))
print("Validation score:", f1_score(Y_test_t, count_model_1.predict(X_test_t)))

Train score: 0.9646610534079036
Validation score: 0.9415650711668969


2) Далее попробуем провести стемминг английских слов в titl'ах. Не забываем, что стоп-слова так же нужно стеммить.


In [ ]:
stemmer_en = nltk.stem.SnowballStemmer('english')
stop_words_ru_2 = nltk.corpus.stopwords.words('russian')
stop_words_en_2 = list(set([stemmer_en.stem(word) for word in nltk.corpus.stopwords.words('english')]))
stop_words_2 = stop_words_ru_2 + stop_words_en_2

def tokenize_2(text):
    return [stemmer_en.stem(word) for word in re.findall(TOKEN_PATTERN, text.lower())]

count_model_2 = Pipeline([
    (
        'vectorizer',
        CountVectorizer(
            stop_words=stop_words_2, ngram_range=(1, 2), token_pattern=None,
            min_df=3, max_df=0.8, tokenizer=tokenize_2
        )
    ),
    ('clf', SGDClassifier(random_state=SEED, loss='log_loss', class_weight='balanced'))
])
count_model_2.fit(x_train_t, y_train_t)
print("Train score:", f1_score(y_train_t, count_model_2.predict(x_train_t)))
print("Validation score:", f1_score(Y_test_t, count_model_2.predict(X_test_t)))

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['r', 'v'] not in stop_words.
  warnings.warn(


Train score: 0.9647001043649089
Validation score: 0.9411199807934699


Понятно, что стемминг английских слов немного ухудшил ситуацию. Поэтому не имеет смысла проводить лемматизацию английских слов. Таким образом, в будущих моделях английские слова будем оставлять, как есть.

3) Далее попробуем провести стемминг русских слов в titl'ах.


In [ ]:
stemmer_ru = nltk.stem.SnowballStemmer('russian')
stop_words_en_3 = nltk.corpus.stopwords.words('english')
stop_words_ru_3 = list(set([stemmer_ru.stem(word) for word in nltk.corpus.stopwords.words('russian')]))
stop_words_3 = stop_words_ru_3 + stop_words_en_3

def tokenize_3(text):
    return [stemmer_ru.stem(word) for word in re.findall(TOKEN_PATTERN, text.lower())]

count_model_3 = Pipeline([
    (
        'vectorizer',
        CountVectorizer(
            stop_words=stop_words_3, ngram_range=(1, 2), token_pattern=None,
            min_df=3, max_df=0.8, tokenizer=tokenize_3
        )
    ),
    ('clf', SGDClassifier(random_state=SEED, loss='log_loss', class_weight='balanced'))
])
count_model_3.fit(x_train_t, y_train_t)
print("Train score:", f1_score(y_train_t, count_model_3.predict(x_train_t)))
print("Validation score:", f1_score(Y_test_t, count_model_3.predict(X_test_t)))

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['оп'] not in stop_words.
  warnings.warn(


Train score: 0.9647202690308775
Validation score: 0.945566912289042


Стемминг русских слов улучшил ситуацию. Значит, и в дальнейшем будем все русские слова стеммить. Но перед этим нужно проверить, может лемматизация ещё больше даст прирост?

4) Как правило, русские слова лучше поддаются лемматизации. Попробуем, может и здесь сработает это правило. Правда придётся считать дольше...

In [ ]:
lemmatizer_ru = pymorphy2.MorphAnalyzer()
stop_words_en_4 = nltk.corpus.stopwords.words('english')
stop_words_ru_4 = list(set([lemmatizer_ru.parse(word)[0].normal_form for word in nltk.corpus.stopwords.words('russian')]))
stop_words_4 = stop_words_ru_4 + stop_words_en_4

def tokenize_4(text):
    return [lemmatizer_ru.parse(word)[0].normal_form for word in re.findall(TOKEN_PATTERN, text.lower())]

count_model_4 = Pipeline([
    (
        'vectorizer',
        CountVectorizer(
            stop_words=stop_words_4, ngram_range=(1, 2), token_pattern=None,
            min_df=3, max_df=0.8, tokenizer=tokenize_4
        )
    ),
    ('clf', SGDClassifier(random_state=SEED, loss='log_loss', class_weight='balanced'))
])
count_model_4.fit(x_train_t, y_train_t)
print("Train score:", f1_score(y_train_t, count_model_4.predict(x_train_t)))
print("Validation score:", f1_score(Y_test_t, count_model_4.predict(X_test_t)))

Train score: 0.9645303326810176
Validation score: 0.945201404176831


Видно, что лемматизация в сравнении со стеммингом особо не улучшила ситуацию, скорее даже наоборот. Оно и понятно, многие русские слова на сайтах (18+) написаны как попало. Естественно, к ним тяжело подобрать лемму. Поэтому лучше для дальнейшнго анализа оставить только стемминг русских слов. Да и стемминг быстрее работает. Это облегчит нам жизнь при финальной кросс-валидации, так как она очень много времени выполняется.

**Разбор URL.**

5) Очевидно, что некоторую информацию мы можем взять из url сайта, так как там тоже довольно часто могут содержаться корни слов, в 99% случаев относящихся к контенту (18+).


а) Первое, что можно сделать, это разделить адрес сайта на домены.
Далее можно убрать домен после последней точки, так как зачастую там для нас нет никакой полезной информации. А шум от этого будет мешать модели. Как обычные сайты, так и сайты (18+) могут заканчиваться на .ru, .com.
Конечно, если этот последний домен содержит в себе корень, характерный для (18+), то его нужно оставлять. **Список этих корней формируется заранее.**


б) Так же мы можем помочь CountVectorizer убрать лишний шум, если после разделения на домены уберём все слова, состоящие из одного символа. Они могут так же быть у любых сайтов, и это так же будет создавать лишний шум. Так же кстати можно поступить и в titl'ах.


в) После всех манипуляций мы получаем слова, разделённые пробелами, длина которых не меньше двух символов. Что ещё можно сделать? Будем делить каждое слово URL'а на 4-граммы. Если длина слова < 4, то оставляем его как есть. Это нам поможет, так как есть корни слов (18+), которые используются в URL. Например, корень '#$%^' спрятан в названии домена '#$%^&365.expert'. Длина корня '#$%^' равна 4-м символам, а это и есть 4-грамма. По моему опыту пользования интернетом очевидно, что среди нормальных сайтов подобных корней практически не встречается.

г) Что же ещё можно сделать? Пусть после шага (в) мы получили представление URL wow-#$%videos.com следующего вида:

'wow #$%v $%vi %vid vide ideo deos'

Видно, что в 4-грамме '#$%v' спрятана 3-грамма '#$%', которая входит во многие слова, являющиеся в большинстве случаев частью контента (18+). Поэтому нужно пройтись по всем 4-граммам и найти в них такие 3-граммы. При нахождении удаляем найденную 4-грамму и вставляем 3-грамму, содержащуюся в ней. **Список таких 3-грамм так же формируется заранее по ручному анализу данных.**

После обработки 'wow #$%v $%vi %vid vide ideo deos' получаем:

'wow #$%v $%vi %vid vide ideo deos'

Вот это уже достаточно хорошо подготовленная строка для CountVectorizer. Осталось полученную строку URL объединить с title.
Как будет реализован код, смотрим ниже.



In [ ]:
#далее идут 3 и 4 граммные маркеры, которые настолько редко используются в контенте не (18+),
#что их можно считать весомыми признаками контента (18+) и выделять отдельно для обучения
#эти маркеры подгпружаются из файла 'markers.json'. Напрямую я их не могу выложить в публичный доступ,
#за доступом обращайтесь ко мне в Telegram @crolzx
import json
with open('markers.json', 'r') as f:
    markers = json.load(f)
OBVIOUS_URL_4MARKERS = markers['url_4'] #4-граммы маркеры, которые в 99% случаев встречаются в url сайтов 18+ (нужны для обработки концов URL)
OBVIOUS_URL_3MARKERS = markers['url_3'] #3-граммы маркеры, которые в 99% случаев встречаются в url сайтов 18+
OBVIOUS_TITLE_3MARKERS = markers['title_3'] #3-граммы маркеры, которые в 99% случаев встречаются в title сайтов 18+
OBVIOUS_URL_MARKERS = OBVIOUS_URL_4MARKERS + OBVIOUS_URL_3MARKERS

def get_ngrams(text, n):
  n_grams = nltk.ngrams(text, n)
  return [''.join(grams) for grams in n_grams]

def GetListSubStrs(strList, flag):
  if flag == 0:
    MARKERS = OBVIOUS_URL_3MARKERS
  else:
    MARKERS = OBVIOUS_TITLE_3MARKERS
  markerList = []
  strListLen = len(strList)
  for marker in MARKERS:
    i = 0
    while i < strListLen:
      if marker in strList[i]:
        markerList.append(marker)
        del strList[i]
        strListLen-=1
        continue
      i+=1
  return list(set(markerList))

def GetStopWords(lstStpWords):
  lst_words_ngrams = []
  for word in lstStpWords:
    if len(word) <= 3 and len(word) >= 1:
      lst_words_ngrams.append(word)
  return lst_words_ngrams

def GetMarkersInWord(word):
  markerList = []
  for marker in OBVIOUS_URL_MARKERS:
    if marker in word:
      markerList.append(marker)
  return markerList

def GetURLTrainIm(x_url):
  lst_url_splited = []
  for text_url in x_url:
    lst_words_ngrams = []
    tempLst = re.findall(TOKEN_PATTERN, text_url.lower())
    if len(tempLst) > 0:
      lstMarkers = GetMarkersInWord(tempLst[-1])
      del tempLst[-1]
      tempLst+=lstMarkers
    for word in tempLst:
      if (len(word) < 4):
        lst_words_ngrams.append(word)
        continue
      tempLstGrams = get_ngrams(word, 4)
      tempLstGrams = GetListSubStrs(tempLstGrams, 0) + tempLstGrams
      lst_words_ngrams.append(' '.join(tempLstGrams))
    lst_url_splited.append(' '.join(lst_words_ngrams))
  return lst_url_splited

#обучающая выборка (подготовленные URL-адреса и пока не подготовленные titl'ы):
x_train_url = GetURLTrainIm(train_df["url"].values.astype('U'))
x_train_title = train_df["title"].values.astype('U')
#объединение подготовленного для обучения образа URL и его titl'а:
x_train_all = [' '.join([im_url, text_title]) for im_url, text_title in zip(x_train_url, x_train_title)]
y_train_all = train_df["label"].values
#генерируем обучающую и тестовую выборки на основании полученных после объединения текстов
x_train, X_test, y_train, Y_test = train_test_split(x_train_all, y_train_all, test_size=0.5, random_state=SEED)

In [ ]:
stop_words_en_5 = nltk.corpus.stopwords.words('english')
stemmer_ru = nltk.stem.SnowballStemmer('russian')
stop_words_ru_5 = list(set([stemmer_ru.stem(word) for word in nltk.corpus.stopwords.words('russian')]))
stop_words_5 = stop_words_ru_5 + stop_words_en_5

def tokenize_5(text):
    return [stemmer_ru.stem(word) for word in re.findall(TOKEN_PATTERN, text.lower())]

count_model_5 = Pipeline([
    (
        'vectorizer',
        CountVectorizer(
            tokenizer=tokenize_5, stop_words=stop_words_5, ngram_range=(1, 2), token_pattern=None,
            min_df=3, max_df=0.8
        )
    ),
    ('clf', SGDClassifier(random_state=SEED, loss='log_loss', class_weight='balanced'))
])

count_model_5.fit(x_train, y_train)
print("Train score:", f1_score(y_train, count_model_5.predict(x_train)))
print("Validation score:", f1_score(Y_test, count_model_5.predict(X_test)))

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['оп'] not in stop_words.
  warnings.warn(


Train score: 0.9920272655346601
Validation score: 0.9775068071504677


Видно, что ситуация после обработки URL и его объединения с titl'ом значительно улучшилась.

6) Почему бы не попробовать аналогичным образом обработать titl'ы. В коде ниже обработка titl'ов будет в токенайзере. Единственный вопрос состоит в том, что делать со стоп-словами. Ведь после разделения текста на 4-grammы получается и их надо делить на 4-граммы.

Не совсем... Когда мы делим текст на 4-граммы, то некоторые 4-граммы длинных слов могут совпадать с некоторыми 4-граммами стоп-слов, что не хорошо. Выход состоит в том, чтобы оставить в стеммированных стоп-словах только стоп-слова длиной 1, 2 и 3. В самом тексте же слова из 1, 2 и 3 символа не будут делиться на 4 граммы, поэтому стоп-слова меньшей длины так и останутся стоп-словами. И короткие слова так и останутся короткими словами.

Ниже полностью работоспособный код, который улучшает предсказания ещё больше.

In [ ]:
stop_words_en_6 = GetStopWords(nltk.corpus.stopwords.words('english'))
stemmer_ru = nltk.stem.SnowballStemmer('russian')
stop_words_ru_6 = GetStopWords(list(set([stemmer_ru.stem(word) for word in nltk.corpus.stopwords.words('russian')])))
stop_words_6 = stop_words_ru_6 + stop_words_en_6

def tokenize_6(text):
    lst = re.findall(TOKEN_PATTERN, text.lower())
    lstLen = len(lst)
    i = 0
    while i < lstLen:
      if len(lst[i]) < 2:
        del lst[i]
        lstLen-=1
        continue
      i+=1
    lst = [stemmer_ru.stem(word) for word in lst]
    n_gramLst = []
    for word in lst:
      if (len(word) < 4):
        n_gramLst.append(word)
        continue
      tempLst = get_ngrams(word, 4)
      tempLst = GetListSubStrs(tempLst, 1) + tempLst
      n_gramLst += tempLst
    return list(set(n_gramLst))

count_model_6 = Pipeline([
    (
        'vectorizer',
        CountVectorizer(
            tokenizer=tokenize_6, stop_words=stop_words_6, ngram_range=(1, 2), token_pattern=None,
            min_df=3, max_df=0.8
        )
    ),
    ('clf', SGDClassifier(random_state=SEED, loss='log_loss', class_weight='balanced'))
])

count_model_6.fit(x_train, y_train)
print("Train score:", f1_score(y_train, count_model_6.predict(x_train)))
print("Validation score:", f1_score(Y_test, count_model_6.predict(X_test)))

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['оп'] not in stop_words.
  warnings.warn(


Train score: 0.9951332278866043
Validation score: 0.9814771118452099


In [ ]:
count_model_6.fit(x_train_all, y_train_all)
print("Train_all score:", f1_score(y_train_all, count_model_6.predict(x_train_all)))

Train_all score: 0.9921847034960029


In [ ]:
#генерируем контрольную тестовую выборку (для загрузки на портал)
x_test_control_urls = GetURLTrainIm(test_df["url"].values)
x_test_control_title = test_df["title"].values
x_test_control = [' '.join([im_url, text_title]) for im_url, text_title in zip(x_test_control_urls, x_test_control_title)]
res1 = count_model_6.predict(x_test_control)
x_test_control_id = test_df["ID"].values.reshape(165378, 1)
res1 = res1.reshape(165378, 1)
result = np.concatenate((x_test_control_id, res1), axis = 1)
print(result)
np.savetxt("result.csv", result.astype(int), fmt='%i', delimiter=",", header="ID,label", comments='')

[[135309      0]
 [135310      0]
 [135311      0]
 ...
 [300684      0]
 [300685      0]
 [300686      0]]
